In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# One download shared by every export; a re-run reads what is already on disk.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests its images one level down, under the split name.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import HuggingFaceDatasetConfig, SourceConfig, TaskConfig
from dataeval_flow.workflows.data_splitting import DataSplittingConfig

workflow = DataSplittingConfig(
    name="mv_split",
    test_frac=0.2,  # 20% of full dataset held out for test
    val_frac=0.0,  # Must be 0 when num_folds > 1; validation is 1/num_folds
    num_folds=3,  # 3-fold cross-validation
    stratify=True,  # Preserve class distribution in each partition
)

task = TaskConfig(
    name="split_military_vehicles",
    workflow="mv_split",
    sources="mv_src",
)

# Build the pipeline configuration
config = PipelineConfig(
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train"),
    ],
    workflows=[workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/workflows/data_splitting/_workflow.py:173: UserWarning: `height` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  balance_output = Balance().evaluate(metadata)


In [4]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [5]:
print(result.report())


  DATASET SPLITTING: 7823 ITEMS → 3 FOLD(S)
  Timestamp:    2026-09-25T21:31:57.226629+00:00
  Duration:     7.03s
  Source:       mv_src (mv_train)
------------------------------------------------------------------------------------------

  SUMMARY
  -------
  Class distribution (full dataset) .............................................   [..]
  Split sizes across folds .................................... 3 folds, test=1565  [..]
  Class distribution across splits (fold 0 of 3) ................................   [..]
  Stratification quality ................................ OK - max deviation 0.1pp  [ok]
  Pre-split balance (mutual information) ........................................   [..]
  Pre-split diversity ...........................................................   [..]

  Health: All checks passed [ok]

  CLASS DISTRIBUTION (FULL DATASET)
  Max/min class ratio: 3.6:1

  Class       Count
  ----------  -----
  T-72          424  ██████████████████████████████
  BTR-80   

In [6]:
raw = result.output.raw

print(f"Dataset size: {raw.dataset_size}")
print(f"Test indices: {len(raw.test_indices)}")
print(f"Number of folds: {len(raw.folds)}")

Dataset size: 7823
Test indices: 1565
Number of folds: 3


In [7]:
import polars as pl

rows = []
for i, fold in enumerate(raw.folds):
    rows.append({"fold": i, "train": len(fold.train_indices), "val": len(fold.val_indices)})
print(pl.DataFrame(rows))
print(f"\nTest (shared across folds): {len(raw.test_indices)} samples")

shape: (3, 3)
┌──────┬───────┬──────┐
│ fold ┆ train ┆ val  │
│ ---  ┆ ---   ┆ ---  │
│ i64  ┆ i64   ┆ i64  │
╞══════╪═══════╪══════╡
│ 0    ┆ 4172  ┆ 2086 │
│ 1    ┆ 4172  ┆ 2086 │
│ 2    ┆ 4172  ┆ 2086 │
└──────┴───────┴──────┘

Test (shared across folds): 1565 samples


In [8]:
# Verify no overlap between splits and full coverage per fold
test_set = set(raw.test_indices)

for i, fold in enumerate(raw.folds):
    train_set = set(fold.train_indices)
    val_set = set(fold.val_indices)

    assert train_set.isdisjoint(val_set), f"Fold {i}: train/val overlap!"
    assert train_set.isdisjoint(test_set), f"Fold {i}: train/test overlap!"
    assert val_set.isdisjoint(test_set), f"Fold {i}: val/test overlap!"

    total = len(train_set) + len(val_set) + len(test_set)
    assert total == raw.dataset_size, f"Fold {i}: missing indices: {total} != {raw.dataset_size}"

print(f"All {len(raw.folds)} folds verified: no overlap, full coverage.")

All 3 folds verified: no overlap, full coverage.


In [9]:
# Full dataset label stats
if raw.label_stats_full:
    print("Full dataset:")
    print(f"  Classes: {raw.label_stats_full.get('class_count', '?')}")
    print(f"  Per-class counts: {raw.label_stats_full.get('label_counts_per_class', [])}")

# Per-fold and test label stats
for i, fold in enumerate(raw.folds):
    if fold.label_stats_train:
        print(f"\nFold {i} train: {fold.label_stats_train.get('label_counts_per_class', [])}")
    if fold.label_stats_val:
        print(f"Fold {i} val:   {fold.label_stats_val.get('label_counts_per_class', [])}")
if raw.label_stats_test:
    print(f"\nTest:  {raw.label_stats_test.get('label_counts_per_class', [])}")

Full dataset:
  Classes: 24
  Per-class counts: {0: 344, 1: 119, 2: 261, 3: 322, 4: 393, 5: 390, 6: 341, 7: 389, 8: 391, 9: 240, 10: 413, 11: 361, 12: 252, 13: 372, 14: 279, 15: 327, 16: 332, 17: 281, 18: 375, 19: 424, 20: 299, 21: 294, 23: 298, 22: 326}

Fold 0 train: {0: 183, 1: 63, 2: 140, 3: 172, 4: 209, 5: 208, 6: 182, 7: 207, 8: 209, 9: 128, 10: 220, 11: 193, 12: 134, 13: 198, 14: 149, 15: 175, 16: 177, 17: 149, 18: 200, 19: 227, 20: 159, 21: 157, 23: 159, 22: 174}
Fold 0 val:   {0: 92, 1: 32, 2: 69, 3: 86, 4: 105, 5: 104, 6: 91, 7: 104, 8: 104, 9: 64, 10: 110, 11: 96, 12: 68, 13: 99, 14: 74, 15: 87, 16: 89, 17: 75, 18: 100, 19: 113, 20: 80, 21: 78, 23: 79, 22: 87}

Fold 1 train: {0: 183, 1: 64, 2: 139, 3: 172, 4: 209, 5: 208, 6: 182, 7: 208, 8: 208, 9: 128, 10: 220, 11: 193, 12: 135, 13: 198, 14: 148, 15: 175, 16: 177, 17: 150, 18: 200, 19: 226, 20: 160, 21: 156, 23: 159, 22: 174}
Fold 1 val:   {0: 92, 1: 31, 2: 70, 3: 86, 4: 105, 5: 104, 6: 91, 7: 103, 8: 105, 9: 64, 10: 110, 1

In [10]:
# Pre-split balance: mutual information between factors and labels
balance_rows = raw.pre_split_balance.get("balance")
if balance_rows:
    print("Pre-split balance (mutual information):")
    print(pl.DataFrame(balance_rows))
else:
    print("No balance data (dataset may lack metadata factors)")

# Pre-split diversity: Shannon diversity per factor
diversity_rows = raw.pre_split_diversity.get("factors")
if diversity_rows:
    print("\nPre-split diversity:")
    print(pl.DataFrame(diversity_rows))
else:
    print("No diversity data (dataset may lack metadata factors)")

Pre-split balance (mutual information):
shape: (3, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ class_label ┆ 1.0      │
│ height      ┆ 0.008153 │
│ width       ┆ 0.009517 │
└─────────────┴──────────┘

Pre-split diversity:
shape: (3, 3)
┌─────────────┬─────────────────┬──────────────────┐
│ factor_name ┆ diversity_value ┆ is_low_diversity │
│ ---         ┆ ---             ┆ ---              │
│ str         ┆ f64             ┆ bool             │
╞═════════════╪═════════════════╪══════════════════╡
│ class_label ┆ 0.957973        ┆ false            │
│ height      ┆ 0.153298        ┆ true             │
│ width       ┆ 0.169882        ┆ true             │
└─────────────┴─────────────────┴──────────────────┘


In [11]:
meta = result.metadata
print(f"Stratified:  {meta.stratified}")
print(f"Num folds:   {meta.num_folds}")
print(f"Split sizes: {meta.split_sizes}")

Stratified:  True
Num folds:   3
Split sizes: {'train': 4172, 'val': 2086, 'test': 1565}


In [12]:
import json

json_str = result.export(fmt="json")
exported = json.loads(json_str)

# Extract test indices (nested under "raw")
test_idx = exported["raw"]["test_indices"]
print(f"Test indices ({len(test_idx)} samples): {test_idx[:10]}...")

# Extract per-fold train/val indices
for i, fold in enumerate(exported["raw"]["folds"]):
    print(f"Fold {i}: train={len(fold['train_indices'])}, val={len(fold['val_indices'])}")

Test indices (1565 samples): [69, 70, 71, 72, 73, 74, 75, 76, 77, 78]...
Fold 0: train=4172, val=2086
Fold 1: train=4172, val=2086
Fold 2: train=4172, val=2086


In [13]:
from dataeval.data import Indices, View

ds = result.dataset
assert ds is not None

test_ds = View(ds, operations=[Indices(test_idx)])
train_ds = View(ds, operations=[Indices(exported["raw"]["folds"][0]["train_indices"])])
val_ds = View(ds, operations=[Indices(exported["raw"]["folds"][0]["val_indices"])])

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 4172, Val: 2086, Test: 1565
